# Stacked BiLSTM · validation v2

Whitespace BPE → Embedding 128 → 3-layer BiLSTM, hidden 256 per direction → Dropout → Linear 512 → 3.

Train uses only v2 train. Checkpoints are selected on v2 VAL with viterbi_lines and fixed cleanup. Every epoch is backed up to Drive. Run cells in order; export prepares TEST before inference cells below.


## Setup



In [ ]:
import os
import sys
import base64
import subprocess
from pathlib import Path

ENVIRONMENT = "colab" if "google.colab" in sys.modules else "kaggle"
root = Path("/content" if ENVIRONMENT == "colab" else "/kaggle/working")
repo_dir = Path(os.environ.get("UA_NER_REPO_DIR", root / "UA-Locations-NER-Deep-Learning"))

if not repo_dir.exists():
    if ENVIRONMENT == "colab":
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
    else:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    env = os.environ.copy()
    env.update(GIT_CONFIG_COUNT="1", GIT_CONFIG_KEY_0="http.extraheader",
               GIT_CONFIG_VALUE_0=f"AUTHORIZATION: basic {auth}")
    try:
        subprocess.run(["git", "clone", "-b", "stacked",
                        "https://github.com/dkalchenko01/UA-Locations-NER-Deep-Learning.git",
                        str(repo_dir)], env=env, check=True)
    finally:
        del token, auth, env

sys.path.insert(0, str(repo_dir))
print(repo_dir)



In [ ]:
%pip install -q -r "$repo_dir/requirements.txt"



In [ ]:
import importlib
for module in ("src.data_io", "src.metrics", "src.training", "src.bigru_helpers", "src.experiment_tracking"):
    importlib.reload(importlib.import_module(module))

import gc
import shutil
import json
import platform
import uuid
from importlib.metadata import version

import pandas as pd
import torch
from tokenizers import Tokenizer

from src.data_io import (get_data_dir, load_manifest, load_data_splits,
                         save_json, save_predictions, file_sha256)
from src.bigru_helpers import CLEAN_OPTIONS, backup_files, postprocessed_scores
from src.dataset import prepare_ner_records, prepare_inference_records, make_loader, LABEL_TO_ID
from src.model import BiLSTMTagger
from src.metrics import competition_scores
from src.preprocessing import clean_locations_v3, serialize_spans
from src.training import set_seed, train_epoch, predict, save_checkpoint

import wandb
from time import perf_counter
from src.experiment_tracking import start_run, log_epoch, log_evaluation, finish_run


## W&B login


In [ ]:
os.environ["WANDB_BASE_URL"] = "https://api.forge.coreweave.com"
os.environ["WANDB_DISABLE_CODE"] = "true"
os.environ["WANDB_CONSOLE"] = "off"

if ENVIRONMENT == "colab":
    from google.colab import userdata
    wandb.login(key=userdata.get("WANDB_API_KEY"), relogin=True)
else:
    from kaggle_secrets import UserSecretsClient
    wandb.login(key=UserSecretsClient().get_secret("WANDB_API_KEY"), relogin=True)



## Configuration



In [ ]:
config = {
    "model": "bilstm", "architecture": "stacked", "embedding_dim": 128, "hidden_size": 256,
    "num_layers": 3, "dropout": 0.2, "seed": 17, "batch_size": 128,
    "learning_rate": 1e-3, "epochs": 5, "gradient_clip": 1.0,
    "max_padded_tokens": 16384, "prepare_batch_size": 2000,
    "sources": ["uk", "ru"], "optimizer": "Adam",
    "tokenizer": "bpe_whitespace.json", "selection_metric": "uk_comp_f1",
    "label_to_id": LABEL_TO_ID, "max_sequence_length": None,
    "early_stopping": False, "environment": ENVIRONMENT,
    "validation_version": "v2", "manifest_name": "validation_manifest_v2", "metric_version": "kaggle_original_regex_v1",
    "postprocessing_version": "expand_trim_empty",
    "decoding": "viterbi_lines", "clean_options": dict(CLEAN_OPTIONS), "upload_checkpoint": False,
}
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
set_seed(config["seed"])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
run_id = "stacked_" + pd.Timestamp.now(tz="UTC").strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8]
run_dir = root / "outputs" / run_id
run_dir.mkdir(parents=True, exist_ok=False)
splits_dir = repo_dir / "data/splits"
tokenizer_path = repo_dir / "artifacts/tokenizers" / config["tokenizer"]
manifest_path = splits_dir / f"{config['manifest_name']}.parquet"
from google.colab import drive
drive.mount("/content/drive")
backup_dir = Path("/content/drive/MyDrive/UA-Locations-NER/stacked") / run_id
backup_dir.mkdir(parents=True, exist_ok=True)

provenance = {
    "run_id": run_id, "seed": config["seed"], "python": platform.python_version(),
    "git_revision": subprocess.run(["git", "-C", str(repo_dir), "rev-parse", "HEAD"],
                                   capture_output=True, text=True).stdout.strip() or None,
    "git_status": subprocess.run(["git", "-C", str(repo_dir), "status", "--short"],
                                 capture_output=True, text=True).stdout.strip(),
    "packages": {name: version(name) for name in ["torch", "tokenizers", "numpy", "pandas", "pyarrow", "tqdm", "wandb"]},
    "device": str(device), "cuda": torch.version.cuda,
    "gpu": torch.cuda.get_device_name(0) if device.type == "cuda" else None,
    "tokenizer_sha256": file_sha256(tokenizer_path),
    "manifest_sha256": file_sha256(manifest_path),
    "source_sha256": {path.name: file_sha256(path) for path in sorted((repo_dir / "src").glob("*.py"))},
    "notebook_sha256": file_sha256(repo_dir / "notebooks/04_bilstm_stacked.ipynb"),
}
save_json(run_dir / "config.json", config)
save_json(run_dir / "provenance.json", provenance)
backup_files(backup_dir, run_dir / "config.json", run_dir / "provenance.json", tokenizer_path)
print(device, run_dir, "\nDrive:", backup_dir)



## Experiment tracking


In [ ]:
os.environ["WANDB_DIR"] = str(run_dir)
run = start_run(
    model="bilstm", config=config, name=run_id,
    validation_version=config["validation_version"], metric_version=config["metric_version"],
    postprocessing_version=config["postprocessing_version"], repo_dir=repo_dir,
    split_manifest_path=manifest_path,
    tokenizer_path=tokenizer_path, provenance=provenance,
    tags=["stacked", f"layers-{config['num_layers']}", f"hidden-{config['hidden_size']}"]
)
save_json(run_dir / "wandb_run.json", {"id": run.id, "url": run.url,
                                      "path": f"{run.entity}/{run.project}/{run.id}"})
backup_files(backup_dir, run_dir / "wandb_run.json")
print(run.url)



## Data



In [ ]:
tokenizer = Tokenizer.from_file(str(tokenizer_path))
tokenizer.no_padding()
tokenizer.no_truncation()
pad_id = tokenizer.token_to_id("[PAD]")
data_dir = get_data_dir(ENVIRONMENT)
manifest = load_manifest(splits_dir, name=config["manifest_name"])
display(pd.crosstab(manifest["source"], manifest["split"]))

record_sets = {"train": [], "val": [], "holdout": []}
alignment = []
for source in config["sources"]:
    splits = load_data_splits(ENVIRONMENT, splits_dir, sources=(source,),
                              manifest_name=config["manifest_name"])
    for split, target in record_sets.items():
        frame = splits.pop(f"{source}_{split}")
        records, skipped = prepare_ner_records(
            tokenizer, frame, source, split, batch_size=config["prepare_batch_size"])
        target.extend(records)
        alignment.append({"source": source, "split": split, "total": len(frame),
                          "retained": len(records), "alignment_failures": len(skipped)})
        save_json(run_dir / f"{source}_{split}_skipped.json", skipped)
        del frame, records
    del splits
    gc.collect()
train_records, val_records, holdout_records = (
    record_sets["train"], record_sets["val"], record_sets["holdout"])
for split in ("val", "holdout"):
    expected = manifest[(manifest["split"] == split) & manifest["source"].isin(config["sources"])]
    assert {r["text_id"] for r in record_sets[split]} == {
        f"{row.source}_{row.row_id}" for row in expected.itertuples()}
alignment = pd.DataFrame(alignment)
display(alignment)
alignment.to_csv(run_dir / "alignment.csv", index=False)
backup_files(backup_dir, run_dir / "alignment.csv")


In [ ]:
loader_options = {
    "pad_id": pad_id, "batch_size": config["batch_size"], "seed": config["seed"],
    "max_padded_tokens": config["max_padded_tokens"], "pin_memory": device.type == "cuda"
}
train_loader = make_loader(train_records, shuffle=True, **loader_options)
val_loader = make_loader(val_records, **loader_options)


holdout_loader = make_loader(holdout_records, **loader_options)


In [ ]:
print("Train:", len(train_records), "VAL:", len(val_records), "Holdout:", len(holdout_records))


## Model



In [ ]:
model_config = {
    "vocab_size": tokenizer.get_vocab_size(), "pad_id": pad_id,
    "embedding_dim": config["embedding_dim"], "hidden_size": config["hidden_size"],
    "num_layers": config["num_layers"], "dropout": config["dropout"], "num_labels": len(LABEL_TO_ID)
}
set_seed(config["seed"])
model = BiLSTMTagger(**model_config).to(device)
optimizer_class = getattr(torch.optim, config["optimizer"])
optimizer = optimizer_class(model.parameters(), lr=config["learning_rate"])
save_json(run_dir / "model_config.json", model_config)
backup_files(backup_dir, run_dir / "model_config.json")
print(model)
print("Parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))

run.config.update({"model_config": model_config})


## Training



In [ ]:
history, best_score, best_epoch = [], -1, None
best_path = run_dir / "best.pt"
for epoch in range(1, config["epochs"] + 1):
    epoch_started = perf_counter()
    train_loader.batch_sampler.set_epoch(epoch)
    train_loss = train_epoch(model, train_loader, optimizer, device, config["gradient_clip"])
    predictions, val_losses = predict(model, val_loader, device, decoding=config["decoding"])
    scores = postprocessed_scores(val_records, predictions, config["clean_options"])
    raw_scores = competition_scores(val_records, predictions)
    result = {"epoch": epoch, "train_loss": train_loss,
              "epoch_seconds": perf_counter() - epoch_started,
              **{f"{source}_val_loss": value for source, value in val_losses.items()},
              **scores, **{f"raw_{key}": value for key, value in raw_scores.items()}}
    history.append(result)
    pd.DataFrame(history).to_csv(run_dir / "history.csv", index=False)
    epoch_path = run_dir / f"epoch_{epoch:02d}.pt"
    save_checkpoint(epoch_path, model, optimizer, epoch, config, model_config, scores, provenance)
    if scores[config["selection_metric"]] > best_score:
        best_score, best_epoch = scores[config["selection_metric"]], epoch
        save_checkpoint(best_path, model, optimizer, epoch, config, model_config, scores, provenance)
    backup_files(backup_dir, epoch_path, best_path, run_dir / "history.csv")
    log_epoch(run, epoch, result, selection_metric=config["selection_metric"],
              epoch_seconds=result["epoch_seconds"],
              extra_metrics={f"val_{config['validation_version']}/raw/{key}": value
                             for key, value in raw_scores.items()})
    print(result)
    del predictions
display(pd.DataFrame(history))
optimizer.zero_grad(set_to_none=True)
del optimizer
gc.collect()


## Select checkpoint


In [ ]:
checkpoint_path = best_path
checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
model.load_state_dict(checkpoint["model_state_dict"])
selected_epoch = checkpoint["epoch"]
summary = {"selected_epoch": selected_epoch, "checkpoint": checkpoint_path.name,
           "selection_metric": config["selection_metric"], "scores": checkpoint["scores"],
           "decoding": config["decoding"], "clean_options": config["clean_options"]}
save_json(run_dir / "summary.json", summary)
backup_files(backup_dir, run_dir / "summary.json")
print(summary)
del checkpoint


## Export VAL, holdout and TEST with logits

The selected best.pt is loaded above. All three exports use the same weights and tokenizer. Spans and logits are saved before cleanup. Exports, checkpoint.pt, tokenizer.json and metadata.json are written to MyDrive, then uploaded to the shared folder by the following cell.


In [ ]:
export_dir = Path("/content/drive/MyDrive/ner_results") / config["model"] / (
    f"{run_id}_{checkpoint_path.stem}_{config['decoding']}")
export_dir.mkdir(parents=True, exist_ok=False)
test = pd.read_csv(data_dir / "test.csv", usecols=["text_id", "text"])
test_records = prepare_inference_records(tokenizer, test)
test_loader = make_loader(test_records, **loader_options)
for name, records, loader in [
    ("val", val_records, val_loader),
    ("holdout", holdout_records, holdout_loader),
    ("test", test_records, test_loader),
]:
    predictions, _ = predict(model, loader, device, decoding=config["decoding"], return_logits=True)
    save_predictions(export_dir / f"{name}.jsonl", records, predictions)
    del predictions
    gc.collect()
shutil.copy2(checkpoint_path, export_dir / "checkpoint.pt")
shutil.copy2(tokenizer_path, export_dir / "tokenizer.json")
save_json(export_dir / "metadata.json", {
    "checkpoint": "checkpoint.pt", "checkpoint_original_name": checkpoint_path.name,
    "checkpoint_sha256": file_sha256(checkpoint_path), "epoch": selected_epoch,
    "model_config": model_config, "training_config": config, "provenance": provenance,
    "tokenizer_sha256": file_sha256(tokenizer_path),
    "manifest_sha256": file_sha256(manifest_path), "label_to_id": LABEL_TO_ID,
    "decoding": config["decoding"], "postprocessing": "none",
    "val_rows": len(val_records), "holdout_rows": len(holdout_records),
    "test_rows": len(test_records),
})
print("Saved on Drive:", export_dir)


In [ ]:
from google.colab import auth
import google.auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload

def upload_ensemble_package(export_dir, checkpoint_path, tokenizer_path):
    auth.authenticate_user()
    credentials, _ = google.auth.default()
    service = build("drive", "v3", credentials=credentials)
    parent_id = "1TXxyVIo1EtLJN-bXTp2HavZdzlgnDJ5j"

    files = {
        "val.jsonl": export_dir / "val.jsonl",
        "holdout.jsonl": export_dir / "holdout.jsonl",
        "test.jsonl": export_dir / "test.jsonl",
        "metadata.json": export_dir / "metadata.json",
        "checkpoint.pt": Path(checkpoint_path),
        "tokenizer.json": Path(tokenizer_path),
    }

    for path in files.values():
        if not path.is_file():
            raise FileNotFoundError(path)

    def find_item(parent, name):
        escaped = name.replace(chr(92), chr(92) * 2).replace(
            "'", chr(92) + "'"
        )
        result = service.files().list(
            q=f"'{parent}' in parents and name = '{escaped}' and trashed = false",
            fields="files(id)",
            supportsAllDrives=True,
            includeItemsFromAllDrives=True,
        ).execute()
        return next(iter(result.get("files", [])), None)

    folder = find_item(parent_id, export_dir.name)
    if folder is None:
        folder = service.files().create(
            body={
                "name": export_dir.name,
                "mimeType": "application/vnd.google-apps.folder",
                "parents": [parent_id],
            },
            fields="id",
            supportsAllDrives=True,
        ).execute()

    for name, path in files.items():
        media = MediaFileUpload(str(path), resumable=True)
        existing = find_item(folder["id"], name)

        if existing:
            request = service.files().update(
                fileId=existing["id"],
                media_body=media,
                supportsAllDrives=True,
            )
        else:
            request = service.files().create(
                body={"name": name, "parents": [folder["id"]]},
                media_body=media,
                supportsAllDrives=True,
            )

        response = None
        while response is None:
            _, response = request.next_chunk()
        print("Uploaded:", name)

    print(f"https://drive.google.com/drive/folders/{folder['id']}")

upload_ensemble_package(export_dir, checkpoint_path, tokenizer_path)

## Test inference



In [ ]:
test = pd.read_csv(data_dir / "test.csv", usecols=["text_id", "text"])
test_records = prepare_inference_records(tokenizer, test)
test_predictions, _ = predict(model, make_loader(test_records, **loader_options), device)
save_predictions(run_dir / "test_predictions.jsonl", test_records, test_predictions)

for filename, cleaner in [("submission.csv", list), ("submission_clean_v3.csv", clean_locations_v3)]:
    submission = test[["text_id"]].copy()
    submission["locations"] = submission["text_id"].map(
        lambda text_id: json.dumps(cleaner(test_predictions[text_id]["locations"]), ensure_ascii=False))
    submission.to_csv(run_dir / filename, index=False)
    print(run_dir / filename)
backup_files(backup_dir, run_dir / "submission.csv", run_dir / "submission_clean_v3.csv")
display(submission.head())

## Post-processing

In [ ]:
import importlib
import src.decoding
import src.training
import src.postprocessing

importlib.reload(src.decoding)
importlib.reload(src.training)
importlib.reload(src.postprocessing)

from src.training import predict
from src.postprocessing import postprocess

clean_options = config["clean_options"]

viterbi_predictions, _ = predict(
    model,
    make_loader(test_records, **loader_options),
    device,
    decoding=config["decoding"],
)

save_predictions(
    run_dir / "test_predictions_viterbi_lines.jsonl",
    test_records,
    viterbi_predictions,
)

submission = test[["text_id"]].copy()
submission["locations"] = [
    json.dumps(
        postprocess(
            row.text,
            viterbi_predictions[row.text_id]["spans"],
            **clean_options,
        ),
        ensure_ascii=False,
    )
    for row in test.itertuples(index=False)
]

path = run_dir / "submission_viterbi_lines_clean_v5.csv"
submission.to_csv(path, index=False)

save_json(run_dir / "viterbi_lines_config.json", {
    "checkpoint": str(checkpoint_path),
    "decoding": "viterbi_lines",
    "postprocessing": clean_options,
})

backup_files(backup_dir, path, run_dir / "viterbi_lines_config.json")
print(path)
display(submission.head())

In [ ]:
changes = []

for row in test.itertuples(index=False):
    before = postprocess(
        row.text,
        test_predictions[row.text_id]["spans"],
        **clean_options,
    )
    after = postprocess(
        row.text,
        viterbi_predictions[row.text_id]["spans"],
        **clean_options,
    )

    if before != after:
        changes.append({
            "text_id": row.text_id,
            "text": row.text,
            "before": before,
            "after": after,
        })

for change in changes:
    print("text_id:", change["text_id"])
    print(change["text"])
    print("ARGMAX: ", json.dumps(change["before"], ensure_ascii=False))
    print("VITERBI_LINES:", json.dumps(change["after"], ensure_ascii=False))
    print()

pd.DataFrame(changes).to_json(
    run_dir / "viterbi_lines_changes.jsonl",
    orient="records", lines=True, force_ascii=False,
)

print("Changed messages:", len(changes))

## Test chunking

In [ ]:
# import re

# chunk_chars = 200
# test = pd.read_csv(data_dir / "test.csv", usecols=["text_id", "text"])
# chunks = []

# for row in test.itertuples(index=False):
#     for line in re.finditer(r"[^\r\n]+", row.text):
#         start, end = line.span()
#         while start < end:
#             stop = min(start + chunk_chars, end)
#             if stop < end:
#                 space = row.text.rfind(" ", start, stop)
#                 if space > start:
#                     stop = space
#             text = row.text[start:stop]
#             if text.strip():
#                 chunks.append({
#                     "text_id": len(chunks), "text": text,
#                     "original_id": row.text_id, "start": start,
#                 })
#             start = stop

# chunks = pd.DataFrame(chunks)
# chunk_records = prepare_inference_records(tokenizer, chunks)
# chunk_predictions, _ = predict(
#     model, make_loader(chunk_records, **loader_options), device
# )

In [ ]:
# chunked_predictions = {
#     text_id: {"spans": [], "locations": []}
#     for text_id in test["text_id"]
# }

# for row in chunks.itertuples(index=False):
#     chunked_predictions[row.original_id]["spans"].extend(
#         (a + row.start, b + row.start)
#         for a, b in chunk_predictions[row.text_id]["spans"]
#     )

# for row in test.itertuples(index=False):
#     prediction = chunked_predictions[row.text_id]
#     prediction["spans"] = sorted(set(prediction["spans"]))
#     prediction["locations"] = [
#         row.text[a:b] for a, b in prediction["spans"]
#     ]

# chunk_dir = run_dir / f"chunked_{checkpoint_path.stem}_{chunk_chars}"
# chunk_dir.mkdir(exist_ok=True)
# save_json(chunk_dir / "config.json", {
#     "checkpoint": str(checkpoint_path), "chunk_chars": chunk_chars,
#     "split": "lines_then_spaces",
# })

# for filename, cleaner in [
#     ("submission_chunked.csv", list),
#     ("submission_chunked_clean_v3.csv", clean_locations_v3),
# ]:
#     submission = test[["text_id"]].copy()
#     submission["locations"] = submission["text_id"].map(
#         lambda text_id: json.dumps(
#             cleaner(chunked_predictions[text_id]["locations"]),
#             ensure_ascii=False,
#         )
#     )
#     submission.to_csv(chunk_dir / filename, index=False)
#     print(chunk_dir / filename)

# display(submission.head())

## Finish tracking


In [ ]:
finish_run(run, best_epoch=selected_epoch, best_score=best_score,
           selection_metric=config["selection_metric"], checkpoint_path=checkpoint_path,
           upload_checkpoint=config["upload_checkpoint"])
